# Phase 3: Cleaning and Preparing the Data

This is where we take the raw data and turn it into something a model can actually learn from. We'll be filtering the crops, handling missing values, and even pulling in some extra data from Wikipedia to see if a farm's region makes a difference in its yield.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('..')
from src.data_cleaning import filter_target_crops, remove_invalid_rows, handle_missing_values, cap_rainfall_outliers
from src.feature_engineering import create_yield_per_hectare
from src.scraping import fetch_wikipedia_tables
from src.merge_data import merge_supplementary_data

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline


In [2]:
# Load the dataset from Phase 2 (update the path as needed)
# DATA_PATH = 'path/to/your/data.csv'

# df = pd.read_csv(DATA_PATH)
# print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
# df.head()

DATA_PATH = '../data/raw/crop_yield.csv'
try:
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
    display(df.head())
except FileNotFounderror:
    print('error: Dataset not found. Please ensure crop_yield.csv is located in the data/raw/ directory.')


Loaded dataset: 19689 rows x 10 columns


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [3]:
# Focus analysis exclusively on Rice and Wheat crops using src function
df_selected = filter_target_crops(df)
print(f'Shape after filtering for Rice and Wheat(rows,columns): {df_selected.shape}')


Shape after filtering for Rice and Wheat(rows,columns): (1742, 9)


In [4]:
# Remove anomalous records using src function
df_selected = remove_invalid_rows(df_selected)
print(f'Shape after removing invalid records(rows,columns): {df_selected.shape}')


Shape after removing invalid records(rows,columns): (1742, 9)


---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [5]:
print('Missing values before cleaning:')
print(df_selected.isnull().sum())

# Handle missing values using src function
df_clean = handle_missing_values(df_selected)

print(f'Missing values remaining: {df_clean.isnull().sum().sum()}')


Missing values before cleaning:
State              0
Crop               0
Crop_Year          0
Season             0
Area               0
Production         0
Annual_Rainfall    0
Fertilizer         0
Pesticide          0
dtype: int64
Missing values remaining: 0


In [6]:
# Cap extreme rainfall outliers using src function
df_clean = cap_rainfall_outliers(df_clean)


Rainfall outliers capped at 3608.7 mm.


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [7]:
# core project requirement: Create the yield_per_hectare target variable using src function
df_clean = create_yield_per_hectare(df_clean)
display(df_clean[['Crop', 'Production', 'Area', 'yield_per_hectare']].head())


Feature engineering complete: yield_per_hectare added.


,Crop,Production,Area,yield_per_hectare
16,Rice,398311,607358.0,0.655809
17,Rice,209623,174974.0,1.198024
18,Rice,1647296,1743321.0,0.944918
26,Wheat,110054,84698.0,1.299370
51,Rice,2340493,1031530.0,2.268953


In [8]:
# Part of the project is to pull in fresh data via web scraping. 
# I'm saving a checkpoint for the data BEFORE I add the Wikipedia data 
# so we can compare the results later.
import os
os.makedirs('../data/processed', exist_ok=True)
PRE_SCRAPE_PATH = '../data/processed/crop_yield_pre_scraping.csv'
df_clean.to_csv(PRE_SCRAPE_PATH, index=False)
print(f'Pre-scraping checkpoint saved: {df_clean.shape} -> {PRE_SCRAPE_PATH}')

# Scrape the Wikipedia table and then merge into primary dataset
df_wiki = fetch_wikipedia_tables()
display(df_wiki.head() if not df_wiki.empty else 'Failed to scrape')

df_integrated = merge_supplementary_data(df_clean, df_wiki)

# Zone merge quality check
zone_cols = [c for c in df_integrated.columns if 'Zone' in c]
if zone_cols:
    unmatched = df_integrated[zone_cols].apply(lambda r: not any(r), axis=1).sum()
    print(f'Zone merge: {len(df_integrated) - unmatched}/{len(df_integrated)} rows matched ({unmatched} unmatched, {unmatched/len(df_integrated)*100:.1f}% missing zone)')
print(f'Integrated dataset shape: {df_integrated.shape}')


Pre-scraping checkpoint saved: (1742, 10) -> ../data/processed/crop_yield_pre_scraping.csv


Scraping successful. Extracted 29 states/regions from Wikipedia.


,State,Zone
0,Andhra Pradesh,Southern
1,Arunachal Pradesh,North-Eastern
2,Assam,North-Eastern
3,Bihar,Eastern
4,Chhattisgarh,Central


Merging primary dataset with scraped Wikipedia data on [State]...
Merge complete. New dataset shape: (1742, 11)
Zone merge: 1742/1742 rows matched (0 unmatched, 0.0% missing zone)
Integrated dataset shape: (1742, 11)


In [9]:
# Strip trailing whitespace from categorical columns before encoding
# This prevents column names like 'Season_Kharif     ' with trailing spaces
for col in ['Season', 'State', 'Crop', 'Zone']:
    if col in df_integrated.columns:
        df_integrated[col] = df_integrated[col].str.strip()

# Encode categorical variables using one-hot encoding
categorical_columns = ['Crop', 'Season', 'State', 'Zone']
existing_cats = [col for col in categorical_columns if col in df_integrated.columns]
df_integrated = pd.get_dummies(df_integrated, columns=existing_cats, drop_first=True)

# Convert boolean dummy columns to int (0/1) — cleaner output and better sklearn compatibility
bool_cols = df_integrated.select_dtypes(include='bool').columns
df_integrated[bool_cols] = df_integrated[bool_cols].astype(int)
print(f'Categorical variables encoded. Shape: {df_integrated.shape}')


Categorical variables encoded. Shape: (1742, 47)


In [10]:
# Scaling the numeric features so the model doesn't get overwhelmed by different unitsually
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# Note: Annual_Rainfall is the correct column name from the raw dataset
cols_to_scale = ['Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide']
valid_cols_to_scale = [col for col in cols_to_scale if col in df_integrated.columns]

df_integrated[valid_cols_to_scale] = scaler.fit_transform(df_integrated[valid_cols_to_scale])
print(f'Standardized columns: {valid_cols_to_scale}')


Standardized columns: ['Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide']


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [11]:
# Optional: Verify the integrated data
# df_integrated.head()

---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [12]:
# Final check on data types and shape
print('--- Final Data Profile ---')
print(f'Final Shape: {df_integrated.shape}')
print(df_integrated.dtypes.value_counts())

# Verify no nulls remain in the feature set
null_check = df_integrated.isnull().sum().sum()
print(f'\nTotal Null Values: {null_check}')

# Save the processed data for modeling
processed_path = '../data/processed/crop_yield_cleaned.csv'
df_integrated.to_csv(processed_path, index=False)
print(f'\nFinal dataset saved to: {processed_path}')


--- Final Data Profile ---
Final Shape: (1742, 47)
int64      41
float64     6
Name: count, dtype: int64

Total Null Values: 0

Final dataset saved to: ../data/processed/crop_yield_cleaned.csv
